# Week 12 Practice: Clustering and Autoencoders

CSCI1409 Deep Learning · based on the Unsupervised Learning lecture (Clustering + Autoencoders)

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Do not use AI tools for any task (ChatGPT, Claude, Gemini, Copilot, Google AI Overviews).
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.
- `sklearn` is used **only** to load the digits dataset. Write K-means yourself.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Task 1: K-means clustering

We have points but **no labels**. K-means splits them into $K$ groups (clusters). Each cluster has a center called a **centroid** $\mu_k$. The algorithm repeats two steps:

1. **Assign:** give every point to its nearest centroid.
2. **Update:** move every centroid to the mean of the points it got.

It stops when the assignments do not change. The quality of a clustering is measured by the **inertia**, the sum of squared distances from each point to its centroid:

$$\text{inertia} = \sum_{i} \lVert x_i - \mu_{c(i)} \rVert^2$$

where $c(i)$ is the cluster of point $i$. Smaller inertia means tighter clusters.

### 1a) K-means by hand

Six points on a line: $X = [1, 2, 4, 7, 9, 10]$, $K = 2$, starting centroids $\mu_1 = 1$, $\mu_2 = 4$.

1. On paper, do the assign and update steps until nothing changes. For each iteration, write which points go to which cluster and the new centroids. Then compute the final inertia.
2. Write a function `kmeans(X, C)` in PyTorch. `X` has shape `[N, D]` (N points, D features) and `C` has shape `[K, D]` (starting centroids). It should return the labels, the final centroids and the inertia. Hint: `torch.cdist(X, C)` gives all distances as an `[N, K]` matrix, and `.argmin(dim=1)` picks the nearest centroid. Check it on $X$.
3. Run it again with starting centroids $\mu_1 = 9$, $\mu_2 = 10$. Do you get the same final clusters? Does it need the same number of iterations?

In [ ]:
X = torch.tensor([1., 2., 4., 7., 9., 10.]).reshape(-1, 1)   # shape [6, 1]
C0 = torch.tensor([1., 4.]).reshape(-1, 1)                   # shape [2, 1]


def kmeans(X, C, max_iters=100):
    # Your code here
    pass

**Your answer:**

### 1b) How many clusters? The elbow method

K-means needs $K$ from us. One common way to choose it is the **elbow method**: run K-means for $K = 1, 2, \dots$ and plot the inertia. The inertia always goes down, but at some $K$ it stops going down fast. That bend (the "elbow") is a good choice.

```python
torch.manual_seed(0)
centers = torch.tensor([[0., 0.], [5., 5.], [0., 6.]])
data = torch.cat([c + torch.randn(50, 2) for c in centers])   # 150 points in 2D
```

1. Plot `data` with `plt.scatter`. How many groups do you see?
2. For each $K$ from 1 to 6, start from $K$ random data points as centroids (`data[torch.randperm(len(data))[:K]]`) and run your `kmeans`. Print the inertia for each $K$ and plot inertia against $K$. Where is the elbow?
3. Plot the clusters for the $K$ you chose (color = label). Why does the inertia keep going down after the elbow? Why do we not just pick the $K$ with the smallest inertia?

In [ ]:
torch.manual_seed(0)
centers = torch.tensor([[0., 0.], [5., 5.], [0., 6.]])
data = torch.cat([c + torch.randn(50, 2) for c in centers])   # 150 points in 2D

# Your code here

**Your answer:**

## Task 2: Autoencoders

An **autoencoder** is a neural network that learns to copy its input to its output, but through a narrow middle layer:

- the **encoder** squeezes the input $x$ into a small vector $z$, the **latent code** (or bottleneck),
- the **decoder** tries to rebuild $x$ from $z$. We call the result $\hat{x}$.

The loss compares the output with the input itself, for example $\text{MSE}(\hat{x}, x)$. **No labels are needed**, so this is unsupervised learning. Because $z$ is small, the network cannot memorize everything and has to keep only the most important information.

### 2a) Build an autoencoder: shapes and parameters

A `nn.Linear(a, b)` layer has $a \cdot b + b$ parameters (weights plus biases). Here is an autoencoder for 28×28 images (784 pixels) with bottleneck size $L$:

```python
encoder = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, L))
decoder = nn.Sequential(nn.Linear(L, 128), nn.ReLU(), nn.Linear(128, 784), nn.Sigmoid())
```

| Case | $L$ |
|---|---|
| A | 32 |
| B | 2 |
| C | 784 |

1. For each case, predict the shape of `z = encoder(x)` and of `x_hat = decoder(z)` for `x = torch.rand(16, 784)`, and the total number of parameters. How many times smaller is $z$ than $x$?
2. Check with PyTorch.
3. Why is there a `Sigmoid` at the end of the decoder? Why is case C a bad autoencoder, even though it can rebuild the input perfectly?

In [ ]:
x = torch.rand(16, 784)   # 16 images with pixel values between 0 and 1

# Your code here

**Your answer:**

### 2b) Train an autoencoder on handwritten digits

We use the small digits dataset from `sklearn`: 1797 images of 8×8 pixels (64 numbers per image), digits 0 to 9.

```python
from sklearn.datasets import load_digits
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32) / 16   # pixels from 0 to 1, shape [1797, 64]
y = torch.tensor(digits.target)                           # only for plots, NOT for training
```

Use this autoencoder with bottleneck size $L$:

```python
encoder = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, L))
decoder = nn.Sequential(nn.Linear(L, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())
```

1. Write a training loop: Adam with `lr=0.01`, MSE loss between `decoder(encoder(X))` and `X`, 500 steps on the whole dataset at once. Train with $L = 2$ and with $L = 16$ (use `torch.manual_seed(0)` before creating each model). Print the final loss for both.
2. Show 8 original images and their reconstructions for both models (use `.reshape(8, 8)` and `plt.imshow`).
3. Which model rebuilds the digits better, and why? Where in the training did we use the labels `y`?

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32) / 16   # pixels from 0 to 1, shape [1797, 64]
y = torch.tensor(digits.target)                           # only for plots, NOT for training

# Your code here

**Your answer:**

## Stretch (optional)

**A) Fuzzy K-means.** In normal ("hard") K-means each point belongs to one cluster. In **fuzzy K-means** each point gets a membership between 0 and 1 for every cluster, and the memberships add up to 1. With fuzziness $m = 2$, the membership of point $x$ in cluster 1 (of two clusters) is

$$u_1 = \frac{1}{1 + \left(\dfrac{d_1}{d_2}\right)^2}$$

where $d_1, d_2$ are the distances from $x$ to the two centroids. Use the final centroids of 1a (2.333 and 8.667). Compute $u_1$ and $u_2$ for the point 4 and for the point 7. Which point is "more sure" about its cluster?

**B) Clustering in the latent space.** Take the codes `z = encoder(X)` of the $L = 2$ model from 2b and plot them with `plt.scatter`, colored by `y`. Do images of the same digit land near each other? Then run your `kmeans` on the codes with $K = 10$.

In [ ]:
# Your code here